In [1]:
from datasets import load_dataset

import torch
import torch.nn as nn
from transformers import AutoModelForMaskedLM, AutoTokenizer

df = load_dataset("microsoft/ms_marco", "v1.1")

model_type_or_dir = "naver/splade-cocondenser-ensembledistil"

# model = Splade(model_type_or_dir, agg="max")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class SPLADE(nn.Module):
    def __init__(self, model_name='distilbert-base-uncased'):
        super().__init__()
        self.bert_mlm = AutoModelForMaskedLM.from_pretrained(model_name)
    def forward(self, input_ids, attention_mask):
        outputs = self.bert_mlm(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        relu_logits = F.relu(logits)
        relu_logits = relu_logits * attention_mask.unsqueeze(-1)
        pooled, _ = torch.max(relu_logits, dim=1)
        weights = torch.log1p(pooled)
        return weights
    
model = SPLADE().to(device)
model.load_state_dict(torch.load('model.pt'))
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_type_or_dir)
reverse_voc = {v: k for k, v in tokenizer.vocab.items()}

model = model.to(device)

/home/jupyter/.local/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
import csv
import torch
from tqdm import tqdm
from collections import defaultdict
import pickle
import os
import time
from datetime import datetime
import torch.nn.functional as F

reverse_index = defaultdict(list)
batch_size = 32

last_save_time = time.time()
save_interval = 30 * 600
counter = 0

start_position = 0
end_position = 9000000
sline_num = 0
slice_cnt = 3
actual_start = (end_position - start_position) // slice_cnt * sline_num
actual_end = (end_position - start_position) // slice_cnt * (sline_num + 1)

if not os.path.exists("backups"):
    os.makedirs("backups")
TOP_K = 50 

with open("collection.tsv") as fd:
    rd = csv.reader(fd, delimiter="\t", quotechar='"')
    
    batch_docs = []
    batch_ids = []
    total_processed = 0
    
    for row in tqdm(rd):
        counter += 1
        if counter < actual_start:
            continue
            
        batch_ids.append(row[0])
        batch_docs.append(row[1])
        
        if len(batch_docs) == batch_size:
            passage_tokens = tokenizer(batch_docs, return_tensors="pt", truncation=True, 
                                      max_length=512, padding=True)
            
            with torch.no_grad():
                input_ids = passage_tokens['input_ids'].to(device)
                attention_mask = passage_tokens['attention_mask'].to(device)
                batch_reps = model(input_ids, attention_mask)
                
                # --- OPTIMIZATION START ---
                # Вместо обработки всего словаря (30k), берем только топ-K на GPU
                # Это работает мгновенно и спасает от медленного Python-цикла на необученных моделях
                top_weights, top_indices = torch.topk(batch_reps, k=TOP_K, dim=1)
            
            # Итерируемся по документам в батче
            for i, doc_id in enumerate(batch_ids):
                # Извлекаем топ-K для конкретного документа
                current_weights = top_weights[i]
                current_indices = top_indices[i]
                
                # Применяем порог (все равно отсекаем слишком маленькие веса, даже если они в топ-K)
                mask = current_weights > 0.01
                
                # Если после маски ничего не осталось, пропускаем
                if not mask.any():
                    continue
                    
                # Применяем маску и переводим в numpy для записи в словарь
                valid_indices = current_indices[mask].cpu().numpy()
                valid_weights = current_weights[mask].cpu().numpy()
                
                for idx, weight in zip(valid_indices, valid_weights):
                    reverse_index[reverse_voc[idx]].append((doc_id, float(weight)))
            # --- OPTIMIZATION END ---
            
            total_processed += len(batch_docs)
            batch_docs = []
            batch_ids = []
            
            current_time = time.time()
            if current_time - last_save_time >= save_interval or counter > actual_end:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                
                with open(f"backups/reverse_index2_1_{timestamp}.pkl", "wb") as f:
                    pickle.dump(dict(reverse_index), f)
                
                with open(f"backups/progress_{timestamp}.txt", "w") as f:
                    f.write(f"Documents processed: {total_processed}\n")
                    f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
                
                print(f"\nBackup saved at {timestamp} - {total_processed} documents processed")
                last_save_time = current_time
                
                if counter > actual_end:
                    break
    
    
print(f"\nProcessing complete. Total documents processed: {total_processed}")


3000031it [1:29:03, 561.45it/s]


Backup saved at 20260131_151625 - 3000032 documents processed

Processing complete. Total documents processed: 3000032


In [ ]:
import csv

def get_document_text(doc_id, file_path="collection.tsv"):
    doc_id = str(doc_id)
    with open(file_path, "r", encoding="utf-8") as fd:
        rd = csv.reader(fd, delimiter="\t", quotechar='"')
        for row in rd:
            if row[0] == doc_id:
                return row[1]
    return None


Unknown instance spec: Please select VM configuration